In [ ]:
import re
from collections import Counter, defaultdict

import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from gensim.models import Word2Vec
from keras.callbacks import EarlyStopping
from keras.layers import LSTM, Dense, Input, SimpleRNN
from nltk.stem import WordNetLemmatizer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    GRU,
    BatchNormalization,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    SpatialDropout1D,
    TextVectorization,
)
from tensorflow.keras.utils import pad_sequences


In [ ]:
def evaluate_model(model, X_test, y_test):
    
    
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
        "predictions": y_pred
    }

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
train_data = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes"),
    random_state=42)


test_data = fetch_20newsgroups(
    subset="test",
    remove=("headers", "footers", "quotes"),
    random_state=42,
)

In [ ]:
X_train, X_test, y_train, y_test = train_data.data, test_data.data, train_data.target, test_data.target

In [ ]:
print(train_data.target_names[:20])
print(set(train_data.target[:20]))

### Target Encoding

The `20 Newsgroups` dataset contains 20 distinct document categories. Each category has a human-readable name stored in `target_names`, while the corresponding labels stored in `target` are integer-encoded class IDs.

The relationship is based on the **index position** of each class name:

* `target = 0` → `target_names[0]` → `alt.atheism`
* `target = 1` → `target_names[1]` → `comp.graphics`
* `target = 2` → `target_names[2]` → `comp.os.ms-windows.misc`
* ...
* `target = 19` → `target_names[19]` → `talk.religion.misc`

Therefore, each target value represents the index of its corresponding category in `target_names`.

Throughout this project, the machine learning models will use the **integer-encoded target labels**, while `target_names` will be used to interpret and present the predicted classes in their original category names.

> **Note:** The target values are zero-indexed, so the 20 categories are represented by integers from `0` to `19`.


In [ ]:
def custom_standardization(text):
    text = tf.strings.lower(text)
    text = tf.strings.regex_replace(text, r"https?://\S+|www\.\S+", "")
    text = tf.strings.regex_replace(text, r"\S+@\S+", "")
    text = tf.strings.regex_replace(text, r"[^\w\s]", "")
    text = tf.strings.regex_replace(text, r"\d+", "")
    text = tf.strings.regex_replace(text, r"\s+", " ")
    text = tf.strings.strip(text)
    return text

In [ ]:
tokenized_train = [
    custom_standardization(word)
    .numpy()
    .decode("utf-8")
    .split()
    for word in X_train]

In [ ]:
word2vec_cbow = Word2Vec( # returns: Word2Vec<vocab=37525, vector_size=128, alpha=0.025>
    sentences=tokenized_train,
    window= 5,
    vector_size = 128,
    min_count = 2,
    sg = 0,
    seed = SEED,
    epochs = 10,
    workers = 4,  
)

In [ ]:
MAX_TOKENS = 20_000
text_vectorizer = TextVectorization(
    max_tokens = MAX_TOKENS,
    standardize=custom_standardization,
    split = "whitespace",
    output_mode = "int"
)


text_vectorizer.adapt(X_train)
vocabulary = text_vectorizer.get_vocabulary()


In [ ]:
print(word2vec_cbow.wv["computer"].shape) # (128,)
print(type(word2vec_cbow.wv["computer"])) #numpy.ndarray
print(word2vec_cbow.wv.most_similar("computer", topn=10))
print(word2vec_cbow.wv.most_similar("sex", topn=10))
print(word2vec_cbow.wv.most_similar("iran", topn=10))
print(word2vec_cbow.wv.similarity("sex", "dick"))
print(word2vec_cbow.wv.similarity("armenia", "usa"))

In [ ]:
print(word2vec_cbow.wv.vectors.shape) 

In [ ]:
known_words = sum(word in word2vec_cbow.wv.key_to_index for word in vocabulary)


print(f"TextVectorization vocabulary: {len(vocabulary)}")
print(f"Words found in Word2Vec: {known_words}")
print(f"Coverage: {known_words / len(vocabulary):.2%}")

In [ ]:
embedding_matrix = np.zeros((len(vocabulary), word2vec_cbow.vector_size)) # (20000, 128)

for index, word in enumerate(vocabulary):
    if word in word2vec_cbow.wv:
        embedding_matrix[index] = word2vec_cbow.wv[word]
        

In [ ]:
X_train_sequences = text_vectorizer(X_train)
X_test_sequences = text_vectorizer(X_test)

In [ ]:
train_sequence_lengths = tf.reduce_sum(
    tf.cast(X_train_sequences != 0 , tf.int32),
    axis = 1
    ).numpy()

In [ ]:
print(np.percentile(train_sequence_lengths, [25, 50, 75, 90, 95, 99, 100]))

In [ ]:
for max_length in [64, 128, 256, 350, 512, 1024]:
    truncated = np.mean(train_sequence_lengths > max_length) * 100
    print(f"{max_length}: truncated {truncated:.2f}% of documents truncated")

In [ ]:
MAX_SEQUENCE_LENGTH = 256

X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen = MAX_SEQUENCE_LENGTH,
    padding = "post",
    truncating = "post",
)


X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen = MAX_SEQUENCE_LENGTH,
    padding = "post",
    truncating = "post"
)


In [ ]:
VOCAB_SIZE = len(text_vectorizer.get_vocabulary())

In [ ]:
word2vec_rnn_model = Sequential([
    
    
    Input(
          shape=(MAX_SEQUENCE_LENGTH,),
          dtype=tf.int32,
          name="token_input"),
    
    
    
    Embedding(
              input_dim = VOCAB_SIZE,
              output_dim = word2vec_cbow.vector_size,
              mask_zero = True,
              trainable= True,
              weights = [embedding_matrix]),
    
    
    SimpleRNN(
            units=128,
            activation = "tanh",
            name = "simple_rnn"
    ),
    
    Dense(
            units=128,
            activation="relu",
            name= "Dense_128"),
    
      Dense(
            units=64,
            activation="relu",
            name= "Dense_64"),
    
    
    Dropout(
            0.4,
            name="dropout"),
    
    
    Dense(
        units = 20,
        activation="softmax",
        name="classification_output")
    
])
word2vec_rnn_model.summary()

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

word2vec_rnn_model.compile(
    optimizer=optimizer,
    loss= "sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
earlystopping = EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    mode= "min",
    restore_best_weights = True,
    min_delta = 1e-4
)

In [ ]:
history = word2vec_rnn_model.fit(
    X_train_padded,
    y_train,
    batch_size = 64,
    epochs=100,
    callbacks = earlystopping,
    verbose=1,
    validation_split = 0.2)

In [ ]:
evaluate_model(word2vec_rnn_model, X_test_padded, y_test)

In [27]:
word_counts = Counter(word
                      for document in tokenized_train
                      for word in document)


glove_vocabulary = {
    word: index
    for index, (word, count) in enumerate(
        item for item in word_counts.items() if item[1] >= 2
    )
}

print(f"GloVe vocabulary size: {len(glove_vocabulary)}")

GloVe vocabulary size: 37525
